In [ ]:
import pandas as pd
import numpy as np


### **Operador &**

faz a operação de and em todo o vetor para cada posição retornando um novo vetor. O and do python já tenta converter tudo em um unico booleano.

### **df.groupby([colunas], as_index=False)**

Agrupa as linhas de acordo com o vetor de colunas. as_index=False evita que o pandas transforme as colunas da lista em indices e mantem elas como colunas normais... se elas fosse indices não seria mais possivel usar ['coluna'] a não ser que seja usado o metodo .reset_index(). Ao usar o groupBy ele retorna um objeto novo chamado DataframeGroupby ou SerieGroupBy que basicamente armazenam os indices de quais grupos cada linha faz parte. Ao realizar uma operação nesse novo objeto essa operação é realizada isoladamente em cada grupo.

### **.agg()**

Muitas linhas -> 1 Linha por grupo

```py
.agg(
    nome_coluna=("coluna agregada", "operação de agregação"),
    media_de_x=("x", "mean"),
    agg_custom=("x", lambda x: x.max() - x.min())
)
```

Cria agregações para cada grupo e colunas escolhidas caso seja usado em um DataFrameGroupby, caso seja usado em um dataframe normal cria as agregações para o dataframe inteiro para as colunas escolhidas. Caso seja uma Serie cria as agregações para ela mesmo.

### **.transform()**

(Mantém: N linhas -> N linhas / Window Function)

### **.filter()**

(Filtra grupos inteiros / HAVING do SQL)

### **.apply()**

(Flexível: aceita qualquer função complexa)
   

### **round(valor, n)**

Arredonda valor para n casas decimais




### **np.select(condições, valores para cada condição, valor padrão)**

Basicamente um if-elif-else vetorizado eficiente. Se não existe um valor verdadeiro e falso retorna os indices das posições em que a condição é verdadeira

In [ ]:
data = {
    'driver_id': [1, 1, 2, 2, 2, 3, 3, 4],
    'city': ['SP', 'SP', 'RJ', 'RJ', 'RJ', 'SP', 'SP', 'BH'],
    'fare': [25.50, 40.00, 15.75, 60.20, 22.00, 100.00, 35.50, 18.25],
    'trip_date': pd.to_datetime([
        '2026-01-05', '2026-01-12', '2026-01-03', '2026-01-15',
        '2026-01-20', '2026-01-02', '2026-01-18', '2026-01-10'
    ])
}

trips = pd.DataFrame(data)

g_by_driver = trips.groupby('driver_id', as_index=False)

avgs_by_driver = g_by_driver.agg(
    avg=("fare", "mean")
)

sorted_avgs = avgs_by_driver.sort_values(by="avg", ascending=False)

sorted_avgs

fare = 


,driver_id,avg
2,3,67.75
0,1,32.75
1,2,32.65
3,4,18.25


In [ ]:
# CTR por app

import pandas as pd

data = {
    'app_id': [123, 123, 123, 123, 234, 234, 234, 123],
    'event_type': ['impression', 'impression', 'click', 'click', 'impression', 'click', 'impression', 'impression'],
    'timestamp': pd.to_datetime([
        '2022-01-01 10:00:00', '2022-01-02 11:00:00', '2022-01-02 11:30:00', 
        '2022-01-03 12:00:00', '2022-03-01 09:00:00', '2022-03-01 09:05:00', 
        '2022-03-02 10:00:00', '2021-12-31 23:59:59' 
    ])
}

events = pd.DataFrame(data)
print("--- Tabela Original ---")
print(events)

#######################################################

events_2022 = events[(events['timestamp'] >= '2022-01-01') & (events['timestamp'] < '2023-01-01')]

events_2022['is_click'] = events_2022['event_type'] == 'click'
events_2022['is_impression'] = events_2022['event_type'] == 'impression'

resumo = events_2022.groupby(['app_id'], as_index=False).agg(
    clicks=('is_click', 'sum'),
    impressions=('is_impression', 'sum')
)

resumo['ctr'] =  round(100.0 *resumo['clicks'] / resumo['impressions'], 2)

resultado = resumo[['app_id','ctr']]

resultado

--- Tabela Original ---
   app_id  event_type           timestamp
0     123  impression 2022-01-01 10:00:00
1     123  impression 2022-01-02 11:00:00
2     123       click 2022-01-02 11:30:00
3     123       click 2022-01-03 12:00:00
4     234  impression 2022-03-01 09:00:00
5     234       click 2022-03-01 09:05:00
6     234  impression 2022-03-02 10:00:00
7     123  impression 2021-12-31 23:59:59


,app_id,ctr
0,123,100.0
1,234,50.0


### **.rank(method=?,ascending=?)**

È um metodo que pode ser aplicado em Series do pandas e retorna uma série nova com o ranking de cada elemento daquela série em ordem crescente ou decrescente. Tem métodos 'dense' que equivale ao DENSE_RANK() do SQL, ou seja, itens iguais tem mesmo rank. Metodo 'min' é equivalente ao RANK(), ou seja, atribui o menor rank e pula 1 rank e por fim metodo 'first' que é equivalente ao ROW_NUMBER() que apenas enumera as linhas sem levar em consideração se são iguais ou não. 

Mesmo sendo um metodo essencialmente de Series, podemos usar diretamente em um Dataframe mas oq retorna é uma matriz com os ranks individuais de cada coluna do Dataframe. Se rank() for usado com DataframesGroupBy (.groupby('x')['y','z']) ou SerieGroupBy('x')['y'], ele fica equivalente ao PARTITION BY do SQL e cria um rank para cada grupo.

### **.sort_values(by=[?,?], ascending=[?,?])**

Ordena uma Série ou Dataframe por um conjunto de colunas no primeiro vetor e sua ordem definida pelo segundo vetor.



In [ ]:
# Top 2 Produtos por Categoria

import pandas as pd

data = {
    'category': ['appliances', 'appliances', 'appliances', 'electronics', 'electronics', 'electronics', 'electronics'],
    'product': ['refrigerator', 'washing machine', 'microwave', 'vacuum cleaner', 'wireless headset', 'phone', 'smartwatch'],
    'spend': [1200.0, 950.0, 400.0, 300.0, 250.0, 1000.0, 800.0]
}

product_spend = pd.DataFrame(data)
print("--- Tabela Original ---")
print(product_spend)

product_spend['rank'] = product_spend.groupby('category')['spend'].rank(method='dense', ascending=False)
top2 = product_spend['rank'] <= 2

product_spend = product_spend[top2].sort_values(by=['category', 'rank'], ascending= [True, True])

print(product_spend)

--- Tabela Original ---
      category           product   spend
0   appliances      refrigerator  1200.0
1   appliances   washing machine   950.0
2   appliances         microwave   400.0
3  electronics    vacuum cleaner   300.0
4  electronics  wireless headset   250.0
5  electronics             phone  1000.0
6  electronics        smartwatch   800.0
      category          product   spend  rank
0   appliances     refrigerator  1200.0   1.0
1   appliances  washing machine   950.0   2.0
5  electronics            phone  1000.0   1.0
6  electronics       smartwatch   800.0   2.0
